# Module 10: Machine Learning with Scikit-Learn
# Lab 03: Supervised Learning — Classification

**Prepared by Information Tech Consultants Ltd**

---

## 🎯 Learning Objectives
By the end of this notebook, you will be able to:
- [ ] Train classification models: Logistic Regression, KNN, Decision Trees, Random Forest
- [ ] Evaluate classifiers using accuracy, precision, recall, F1, and confusion matrix
- [ ] Plot and interpret ROC curves and AUC scores
- [ ] Choose the right classifier for different problems

**⏱ Estimated Time:** 90 minutes  
**📋 Prerequisites:** Module 10 Labs 01–02

In [ ]:
# ============================================================
# 📦 Environment Setup — Run this cell first!
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, roc_auc_score)

# Load the breast cancer dataset — a classic binary classification problem
data = load_breast_cancer(as_frame=True)
X = data.frame.drop("target", axis=1)
y = data.frame["target"]  # 0 = malignant, 1 = benign

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Data ready!")
print(f"Features: {X.shape[1]} | Train: {len(X_train)} | Test: {len(X_test)}")
print(f"\nTarget distribution:")
print(f"  Benign (1):    {(y == 1).sum()} ({(y == 1).mean()*100:.1f}%)")
print(f"  Malignant (0): {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)")

In [ ]:
# ============================================================
# 🛠 Helper Functions (run once, use throughout)
# ============================================================

from IPython.display import HTML, display

def info_box(title, content, color="#0092D6", bg="#E3F2FD"):
    """Display a styled information callout box."""
    display(HTML(f"""
    <div style="background:{bg};padding:15px;border-left:5px solid {color};
    margin:10px 0;border-radius:4px;font-family:Calibri,Arial,sans-serif;">
    <strong>💡 {title}</strong><br>{content}</div>"""))

def warning_box(title, content):
    """Display a warning callout box."""
    display(HTML(f"""
    <div style="background:#FFF3E0;padding:15px;border-left:5px solid #FF9800;
    margin:10px 0;border-radius:4px;font-family:Calibri,Arial,sans-serif;">
    <strong>⚠️ {title}</strong><br>{content}</div>"""))

def interview_box(question, key_points):
    """Display an interview question callout box."""
    display(HTML(f"""
    <div style="background:#F3E5F5;padding:15px;border-left:5px solid #9C27B0;
    margin:10px 0;border-radius:4px;font-family:Calibri,Arial,sans-serif;">
    <strong>🎯 Interview Question</strong><br><em>"{question}"</em><br><br>
    <strong>Key Points:</strong> {key_points}</div>"""))

def success_box(content):
    """Display a success/best practice box."""
    display(HTML(f"""
    <div style="background:#E8F5E9;padding:15px;border-left:5px solid #4CAF50;
    margin:10px 0;border-radius:4px;font-family:Calibri,Arial,sans-serif;">
    <strong>✅ Best Practice</strong><br>{content}</div>"""))

def exercise_header(num, title, difficulty="⭐"):
    """Display a formatted exercise header."""
    display(HTML(f"""
    <div style="background:#E8EAF6;padding:15px;border-left:5px solid #0092D6;
    margin:15px 0;border-radius:4px;font-family:Calibri,Arial,sans-serif;">
    <strong>🏋️ Exercise {num}: {title}</strong> | Difficulty: {difficulty}</div>"""))

def draw_pipeline(steps, arrow="→"):
    """Draw a simple pipeline flow diagram."""
    flow = f" {arrow} ".join([f"[{s}]" for s in steps])
    display(HTML(f"""
    <div style="background:#F5F5F5;padding:20px;border-radius:8px;
    text-align:center;font-family:monospace;font-size:16px;margin:10px 0;">
    {flow}</div>"""))

print("✅ Helper functions loaded!")

## 🚀 Complete Working Example

Let's train five different classifiers and compare them.

In [ ]:
# ============================================================
# 🚀 COMPLETE WORKING EXAMPLE — Run me first!
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=5000, random_state=42),
    "KNN (k=5)":           KNeighborsClassifier(n_neighbors=5),
    "Decision Tree":       DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

results = {}
print(f"{'Model':<25} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("=" * 70)

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results[name] = {"acc": acc, "prec": prec, "rec": rec, "f1": f1, "model": model}
    print(f"{name:<25} {acc:10.4f} {prec:10.4f} {rec:10.4f} {f1:10.4f}")

print("\n🎉 All classifiers trained!")

---
## 📖 Section 1: Logistic Regression

**What:** Despite its name, Logistic Regression is a *classification* algorithm. It estimates the probability that a data point belongs to a particular class.

**Why it matters:** It's fast, interpretable, and works surprisingly well for many problems. It's often the first classifier you should try.

**Analogy:** Imagine a spam filter. Logistic Regression looks at features of an email (certain words, sender info, etc.) and calculates the *probability* it's spam. If the probability is above 50%, it labels it as spam.

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=5000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Logistic Regression can give probabilities, not just labels!
y_pred = log_reg.predict(X_test_scaled)           # Hard prediction: 0 or 1
y_proba = log_reg.predict_proba(X_test_scaled)     # Probabilities for each class

print("First 5 predictions:")
print(f"{'Actual':<10} {'Predicted':<12} {'P(Malignant)':<15} {'P(Benign)':<15}")
print("=" * 52)
for i in range(5):
    print(f"{y_test.iloc[i]:<10} {y_pred[i]:<12} {y_proba[i][0]:<15.4f} {y_proba[i][1]:<15.4f}")

info_box(
    "Probabilities vs Hard Predictions",
    "<code>predict()</code> gives you the class label (0 or 1).<br>"
    "<code>predict_proba()</code> gives the probability for each class.<br>"
    "Probabilities are more useful — you can adjust the threshold to trade off "
    "precision and recall."
)

---
## 📖 Section 2: Understanding Classification Metrics

In [ ]:
# The Confusion Matrix — the foundation of all classification metrics
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(7, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Malignant", "Benign"])
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix — Logistic Regression", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Explain each quadrant
tn, fp, fn, tp = cm.ravel()
print(f"True Negatives  (TN): {tn} — correctly predicted malignant")
print(f"False Positives (FP): {fp} — predicted benign but was malignant ⚠️")
print(f"False Negatives (FN): {fn} — predicted malignant but was benign")
print(f"True Positives  (TP): {tp} — correctly predicted benign")

warning_box(
    "In Medical Diagnoses, False Negatives Are Dangerous!",
    "A False Negative here means telling someone they don't have cancer when they do. "
    "In such cases, we care more about <b>Recall</b> (catching all actual positives) "
    "than Precision."
)

In [ ]:
# Detailed classification report
print("Full Classification Report:")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=["Malignant", "Benign"]))

info_box(
    "Metrics Explained Simply",
    "<b>Accuracy</b>: Of all predictions, how many were correct?<br>"
    "<b>Precision</b>: Of all predicted positive, how many were actually positive?<br>"
    "<b>Recall</b>: Of all actual positives, how many did we find?<br>"
    "<b>F1 Score</b>: The harmonic mean of precision and recall — a balanced metric."
)

---
## 📖 Section 3: K-Nearest Neighbours (KNN)

**What:** KNN classifies a data point by looking at its *k closest neighbours* and taking a majority vote.

**Analogy:** If you move to a new neighbourhood and want to predict the average income, you'd look at your nearest neighbours. That's KNN!

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# Try different values of k
print(f"{'k':<5} {'Accuracy':>10}  {'F1':>8}")
print("=" * 28)

k_values = [1, 3, 5, 7, 9, 15, 25]
accuracies = []

for k in k_values:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred_k = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred_k)
    f1 = f1_score(y_test, y_pred_k)
    accuracies.append(acc)
    print(f"{k:<5} {acc:10.4f}  {f1:8.4f}")

# Plot k vs accuracy
plt.figure(figsize=(8, 4))
plt.plot(k_values, accuracies, "o-", color="#0092D6", linewidth=2, markersize=8)
plt.xlabel("k (number of neighbours)", fontsize=12)
plt.ylabel("Accuracy", fontsize=12)
plt.title("KNN: Effect of k on Accuracy", fontsize=14, fontweight="bold")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

warning_box(
    "KNN is Sensitive to Feature Scaling!",
    "Since KNN uses distance to find neighbours, features with larger scales "
    "will dominate. <b>Always scale your data before using KNN.</b>"
)

---
## 📖 Section 4: Decision Trees & Random Forests

**What:** Decision Trees split data based on questions about features, creating a flowchart-like structure. Random Forests combine many trees for better results.

**Analogy:** A Decision Tree is like a game of 20 questions: "Is the tumour larger than 15mm? If yes, go left. Is the texture smooth? If yes, go right…" until you reach an answer.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Train a small tree we can visualise
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_train_scaled, y_train)

# Visualise the tree
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(dt, feature_names=X.columns, class_names=["Malignant", "Benign"],
          filled=True, rounded=True, ax=ax, fontsize=9)
ax.set_title("Decision Tree (max_depth=3)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

y_pred_dt = dt.predict(X_test_scaled)
print(f"Decision Tree Accuracy: {accuracy_score(y_test, y_pred_dt):.4f}")

In [ ]:
# Random Forest = many trees combined (ensemble method)
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred_rf = rf.predict(X_test_scaled)

print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")

# Feature importance — which features matter most?
importances = rf.feature_importances_
top_indices = np.argsort(importances)[-10:]  # Top 10

plt.figure(figsize=(10, 5))
plt.barh(range(10), importances[top_indices], color="#0092D6", edgecolor="white")
plt.yticks(range(10), X.columns[top_indices], fontsize=11)
plt.xlabel("Feature Importance", fontsize=12)
plt.title("Top 10 Feature Importances — Random Forest", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

interview_box(
    "What is the difference between a Decision Tree and a Random Forest?",
    "A Decision Tree is a single tree that can easily overfit.<br>"
    "A Random Forest trains <b>many trees on random subsets</b> of the data "
    "and features, then takes a <b>majority vote</b>. This is called 'bagging' "
    "(Bootstrap Aggregating) and it reduces overfitting significantly."
)

---
## 📖 Section 5: ROC Curves & AUC

In [ ]:
# ROC Curve: shows the trade-off between True Positive Rate and False Positive Rate
fig, ax = plt.subplots(figsize=(8, 6))

model_list = {
    "Logistic Regression": LogisticRegression(max_iter=5000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
}

colors = ["#0092D6", "#333333", "#8ACAE7"]

for (name, model), color in zip(model_list.items(), colors):
    model.fit(X_train_scaled, y_train)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f"{name} (AUC = {auc:.4f})")

ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Random Classifier (AUC = 0.5)")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — Model Comparison", fontsize=14, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

info_box(
    "Understanding AUC-ROC",
    "<b>AUC = 1.0</b>: Perfect classifier<br>"
    "<b>AUC = 0.5</b>: No better than random guessing<br>"
    "<b>AUC > 0.9</b>: Excellent model<br>"
    "AUC-ROC is especially useful for <b>imbalanced datasets</b> where accuracy alone is misleading."
)

---
## 🏋️ Exercises

In [ ]:
exercise_header(1, "Explore the Confusion Matrix", "⭐")

# Run this cell and answer the questions below
from sklearn.tree import DecisionTreeClassifier

dt_deep = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_deep.fit(X_train_scaled, y_train)
y_pred_deep = dt_deep.predict(X_test_scaled)

print("Classification Report — Deep Decision Tree:")
print(classification_report(y_test, y_pred_deep, target_names=["Malignant", "Benign"]))

# ❓ Questions:
# 1. What is the recall for Malignant tumours? Why does this matter?
# 2. Is precision or recall more important in cancer detection? Why?
# 3. Compare this to the max_depth=3 tree — which is better?

In [ ]:
exercise_header(2, "Find the Best k for KNN", "⭐⭐")

# TODO: Test k values from 1 to 30 and plot accuracy vs k
# Find the k that gives the best F1 score

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score

k_range = range(1, 31)
f1_scores = []

for k in k_range:
    # YOUR CODE HERE:
    # 1. Create KNeighborsClassifier with n_neighbors=k
    # 2. Fit on training data
    # 3. Predict on test data
    # 4. Calculate F1 score and append to f1_scores
    pass

# Uncomment when ready:
# best_k = k_range[np.argmax(f1_scores)]
# print(f"Best k = {best_k} with F1 = {max(f1_scores):.4f}")

In [ ]:
exercise_header(3, "Build a Complete Classifier", "⭐⭐⭐")

# Challenge: Use the wine dataset. Train at least 3 different classifiers.
# Create a confusion matrix for the best one. Note: this is a MULTI-CLASS problem!

from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
X_wine = wine.frame.drop("target", axis=1)
y_wine = wine.frame["target"]  # Classes: 0, 1, 2

# YOUR CODE HERE:
# 1. Split into train/test
# 2. Scale features
# 3. Train Logistic Regression, KNN, and Random Forest
# 4. Print classification report for each
# 5. Plot confusion matrix for the best model

---
## 📋 Solutions

<details>
<summary>Click to expand Exercise 2 solution</summary>

```python
f1_scores = []
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred_k = knn.predict(X_test_scaled)
    f1_scores.append(f1_score(y_test, y_pred_k))

best_k = list(k_range)[np.argmax(f1_scores)]
print(f"Best k = {best_k} with F1 = {max(f1_scores):.4f}")

plt.plot(k_range, f1_scores, 'o-', color="#0092D6")
plt.xlabel("k"); plt.ylabel("F1 Score"); plt.title("KNN: k vs F1 Score")
plt.show()
```

</details>

<details>
<summary>Click to expand Exercise 3 solution</summary>

```python
X_tr, X_te, y_tr, y_te = train_test_split(X_wine, y_wine, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

models = {
    "Logistic Reg": LogisticRegression(max_iter=5000, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
}

for name, model in models.items():
    model.fit(X_tr_s, y_tr)
    y_pred = model.predict(X_te_s)
    print(f"\n{name}:")
    print(classification_report(y_te, y_pred, target_names=wine.target_names))
```

</details>

---
## 🎯 Key Takeaways

1. **Logistic Regression** is simple, fast, and gives probabilities — great first choice
2. **KNN** is intuitive but sensitive to scaling and slow on large datasets
3. **Decision Trees** are interpretable but prone to overfitting
4. **Random Forests** reduce overfitting by combining many trees — often the best out-of-the-box model
5. **Choose metrics wisely** — accuracy alone can be misleading; use precision, recall, F1, and AUC

## ✅ Self-Assessment Checklist
- [ ] I can train and evaluate classification models in scikit-learn
- [ ] I understand the confusion matrix and can interpret it
- [ ] I know when to prioritise precision vs recall
- [ ] I can plot and interpret ROC curves

## 📚 Next Steps
- **Next Lab:** Module 10 Lab 04 — Unsupervised Learning
- **Practice:** Try the Iris dataset (3 classes) with these classifiers

---
*Prepared by Information Tech Consultants Ltd*